# Занятие 3. Циклы и функции

**План занятия**

1. `for`, `enumerate`, `range`
2. Накопитель и счётчик
3. `while` и его типовые формы
4. Вложенные циклы и их цена
5. Функции: контракт, аргументы, возврат
6. Область видимости
7. Домашние задачи

**Теория:** `theory/03_Циклы_и_функции.md`
Т. Гэддис, гл. 4 (с. 179 / PDF 204) и гл. 5 (с. 226 / PDF 251)

---

## 1. `for`

Проходит по элементам коллекции. Индекс при этом не нужен.

In [ ]:
words = ["alpha", "beta", "gamma"]

for word in words:
    print(word, len(word))

Когда индекс нужен, берут `enumerate`.

In [ ]:
for i, word in enumerate(words):
    print(i, word)

print()
for i, word in enumerate(words, start=1):
    print(i, word)

`range` порождает последовательность целых. Правая граница не включается.

In [ ]:
print(list(range(5)))
print(list(range(2, 5)))
print(list(range(0, 10, 2)))
print(list(range(5, 0, -1)))

**Проверьте себя.** Какая из трёх строк ниже даёт `IndexError`,
какая пропускает первый элемент, какая верна?

In [ ]:
items = [10, 20, 30]

for i in range(len(items)):
    print("A", i, items[i])

for i in range(1, len(items)):
    print("B", i, items[i])

# for i in range(len(items) + 1):
#     print("C", i, items[i])

---

## 2. Накопитель и счётчик

In [ ]:
values = [3, -1, 4, -1, 5, 9]

total = 0                 # накопитель создаётся ДО цикла
for value in values:
    total += value

count = 0
for value in values:
    if value > 0:
        count += 1

print("total:", total, "positive:", count)

Накопитель, созданный внутри цикла, обнуляется на каждом шаге.

In [ ]:
for value in values:
    total_broken = 0       # ошибка: обнуление на каждом шаге
    total_broken += value

print("broken:", total_broken, "вместо", total)
print("осталось только последнее значение")

Готовые функции для типовых случаев:

In [ ]:
print(sum(values))
print(max(values), min(values))
print(sum(1 for v in values if v > 0))

Писать цикл руками стоит, когда на каждом шаге происходит что-то ещё.

---

## 3. `while`

`for` применяют, когда число шагов известно заранее, `while` когда неизвестно.

In [ ]:
# сигнальная метка: специальное значение означает конец данных
lines = ["first", "second", "third", "", "after the end"]
position = 0

line = lines[position]              # чтение ДО цикла
while line != "":
    print("got:", line)
    position += 1
    line = lines[position]          # чтение в конце тела

print("stopped at position", position)

Чтение стоит и до цикла, и в конце тела. Иначе первое значение либо
не проверится, либо потеряется.

In [ ]:
# валидация: повторять, пока значение негодное
raw_values = ["-5", "0", "abc", "7"]

for raw in raw_values:
    if raw.lstrip("-").isdigit() and int(raw) > 0:
        print(f"{raw!r:>6} accepted")
        break
    print(f"{raw!r:>6} rejected")

---

## 4. Вложенные циклы

Внутренний цикл прокручивается целиком на каждом шаге внешнего.

In [ ]:
import time

for n in [200, 400, 800, 1600]:
    data = list(range(n))
    start = time.perf_counter()
    pairs = 0
    for a in data:
        for b in data:
            if a + b == 100:
                pairs += 1
    elapsed = time.perf_counter() - start
    print(f"n = {n:>4}  {elapsed:.4f} s")

Данные выросли вдвое, время выросло вчетверо. Это O(n²).

Правило на первое время: увидели цикл в цикле по одним данным, проверьте,
нельзя ли обойтись одним проходом. Занятия 6 и 9 посвящены тому, как это делается.

---

## 5. Функция

In [ ]:
def count_words(text):
    """Возвращает число слов в строке."""
    return len(text.split())

print(count_words("one two three"))
print(count_words(""))
print(count_words.__doc__)

### Аргументы по умолчанию

In [ ]:
def clip(value, low=0, high=100):
    return max(low, min(value, high))

print(clip(150))
print(clip(150, high=200))
print(clip(-5))

Значение по умолчанию вычисляется один раз при определении функции.
Изменяемый объект в этой роли даёт ловушку.

In [ ]:
def bad(item, acc=[]):        # список создаётся ОДИН раз
    acc.append(item)
    return acc

print(bad(1))
print(bad(2))
print(bad(3), "тот же список копит значения между вызовами")

In [ ]:
def good(item, acc=None):
    acc = [] if acc is None else acc
    acc.append(item)
    return acc

print(good(1))
print(good(2))

### `return` против `print`

In [ ]:
def add_bad(a, b):
    print(a + b)          # показывает результат, не отдаёт его

def add_good(a, b):
    return a + b

x = add_bad(2, 3)
y = add_good(2, 3)
print("x =", x, " y =", y)

try:
    print(x * 2)
except TypeError as error:
    print("TypeError:", error)

Функция без `return` возвращает `None`. Ошибка всплывает позже
и в другом месте, чем допущена.

---

## 6. Область видимости

In [ ]:
def compute():
    local_value = 1
    return local_value

print(compute())

try:
    print(local_value)
except NameError as error:
    print("NameError:", error)

In [ ]:
counter = 0

def increment_broken():
    counter = counter + 1        # присваивание создаёт локальное имя
    return counter

try:
    increment_broken()
except UnboundLocalError as error:
    print("UnboundLocalError:", error)

def increment(value):
    return value + 1             # вход и выход через контракт

counter = increment(counter)
print("counter:", counter)

Функция, тайком меняющая внешнее состояние, ломает главное свойство контракта:
по вызову больше нельзя понять, что произошло. Всё нужное передавайте аргументом,
всё произведённое возвращайте.

---

## 7. Собираем программу

Разбор строки на поля с проверкой контракта.

In [ ]:
def parse_record(line, separator=","):
    """Разбирает строку на три поля. Возвращает кортеж или None."""
    fields = line.strip().split(separator)
    if len(fields) != 3:
        return None
    name, year, role = fields
    if not year.strip().isdigit():
        return None
    return name.strip(), int(year), role.strip()


lines = [
    "Pushkin, 1799, poet",
    "Gogol, 1809, writer",
    "Broken line",
    "Someone, about 1800, unknown",
]

parsed = 0
skipped = 0
for line in lines:
    record = parse_record(line)
    if record is None:
        skipped += 1
        print(f"skipped: {line!r}")
    else:
        parsed += 1
        print(f"ok: {record}")

print(f"\nparsed {parsed}, skipped {skipped}")

Функция возвращает `None` при негодном входе, а вызывающий решает,
что с этим делать. Счётчик пропущенных обязателен: без него потеря данных
проходит незаметно. Подробнее на занятии 5.

---

# Домашние задачи

Рассчитаны примерно на 30 минут. Для задач 3 и 4 сначала запишите план
комментариями, потом пишите код.

### Задача 1. Трассировка (без запуска)

Что напечатает код и почему?

In [ ]:
# Мой ответ: ...

# def f(x, acc=[]):
#     acc.append(x)
#     return len(acc)
#
# print(f(1), f(2), f(3))

### Задача 2. Минимум LeetCode

**LeetCode 1342 Number of Steps to Reduce a Number to Zero** плюс письменный разбор.

In [ ]:
def number_of_steps(num):
    # ваш код здесь
    pass

# print(number_of_steps(14), number_of_steps(8), number_of_steps(0))

### Задача 3. Контракт функции

Напишите `word_stats(text)`, возвращающую кортеж из трёх чисел:
всего слов, уникальных слов, длина самого длинного слова.

Пустая строка должна давать `(0, 0, 0)`, а не падать.

In [ ]:
def word_stats(text):
    # ваш код здесь
    pass

# print(word_stats("one two two three"))
# print(word_stats(""))

### Задача 4. Уберите вложенный цикл

Функция ищет числа, встречающиеся в обоих списках. Сейчас она O(n · m).
Перепишите за O(n + m) и замерьте оба варианта на списках по 5000 элементов.

In [ ]:
def common_slow(a, b):
    result = []
    for x in a:
        if x in b:          # здесь спрятан второй цикл
            result.append(x)
    return result

def common_fast(a, b):
    # ваш код здесь
    pass

import random, time
random.seed(0)
a = [random.randint(0, 20000) for _ in range(5000)]
b = [random.randint(0, 20000) for _ in range(5000)]

start = time.perf_counter()
slow = common_slow(a, b)
print(f"slow: {time.perf_counter() - start:.4f} s, found {len(slow)}")

### Задача 5. Подумать (кода не нужно)

Функция `parse_record` из раздела 7 возвращает `None` при негодном входе.
Назовите два других способа сообщить о негодном входе и по одному аргументу
за и против каждого.

### Задача 6. Трек «алгоритмы» (по желанию)

66 Plus One, 509 Fibonacci Number, 202 Happy Number, 1929 Concatenation of Array.

509 решается циклом с двумя накопителями. Запомните её: на занятии 25
мы решим её заново рекурсией.

---

# Итоги

- Накопитель создаётся до цикла. Внутри он обнуляется на каждом шаге.
- `enumerate` вместо `range(len(items))`.
- Правая граница `range` не включается, отсюда ошибки на единицу.
- `while` для случая, когда число шагов заранее неизвестно.
- Цикл в цикле по одним данным даёт O(n²).
- Функция без `return` возвращает `None`.
- Изменяемое значение по умолчанию копит состояние между вызовами.
- Присваивание внутри функции создаёт локальное имя. `global` не нужен.